
<div class='alert alert-info' style='text-align:justify'>

This notebook is used to explore Gaussian Mixture Models monitors (GMM Monitors) and see how the different parameters influence the performance of the monitoring task. A few metrics are computed to briefly compare the different versions of monitors.
</div>

## Getting hands on

In [1]:
from dataset import Dataset
from feature_extractor import FeatureExtractor
from evaluator import Evaluator

from Monitors import GMMMonitor
from Monitors import MahalanobisMonitor

import csv
import os
import pandas as pd
import torch

In [2]:
model = "densenet"
layer = 98

dataset_ID  = "cifar10"
dataset_OOD = "cifar10"

perturbation = None
adver_attack = "fgsm"

In [3]:
batch_size = 100
TORCH_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [4]:
dataset_train = Dataset(dataset_ID, "train", model, batch_size=batch_size)
dataset_test = Dataset(dataset_ID, "test", model, batch_size=batch_size)
dataset_ood = Dataset(dataset_OOD, "test", model, perturbation, adver_attack, batch_size=batch_size)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [5]:
feature_extractor = FeatureExtractor(model, dataset_ID, [layer], TORCH_DEVICE)

c:\Users\MathieuDARIO\Workspace\neural-network-monitoring-benchmark\feature_extractor.py:312: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.

In [6]:
# deep_features_train = feature_extractor.get_features(dataset_train)
# deep_features_test = feature_extractor.get_features(dataset_test)
# deep_features_ood = feature_extractor.get_features(dataset_ood)

features_train, logits_train, softmax_train, \
    preds_train, labels_train = feature_extractor.get_features(dataset_train)
features_test, logits_test, softmax_test, \
    preds_test, labels_test = feature_extractor.get_features(dataset_test)
features_ood, logits_ood, softmax_ood, \
    preds_ood, labels_ood = feature_extractor.get_features(dataset_ood)

In [7]:
eval_oms = Evaluator("oms", is_novelty=(dataset_ID != dataset_OOD))
eval_oms.fit_ground_truth(labels_test, labels_ood, preds_test, preds_ood)

eval_ood = Evaluator("ood", is_novelty=(dataset_ID != dataset_OOD))
eval_ood.fit_ground_truth(labels_test, labels_ood, preds_test, preds_ood)

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
hp_n_components = [5]
hp_t_covariance = ['diag']

# hp_n_components = ['auto_aic', 'auto_bic', 'auto_knee']
# hp_t_covariance = ['full', 'diag', 'tied', 'spherical']

result = {}

for n in hp_n_components:
    for c in hp_t_covariance:
        print(f"... evaluating parameters (n={n}, c={c})")
        
        gmm_monitor = GMMMonitor(dataset_ID, model, layer, n_components=n, t_covariance=c)
        gmm_monitor.fit(features_train[0], preds_train, labels_train, save=True)
        gmm_name = "GMM_%s_%s" % (n, c)

        print(str([gmm.n_components for gmm in gmm_monitor.gmm]))

        scores_test = gmm_monitor.predict(features_test[0], preds_test)
        scores_ood  = gmm_monitor.predict(features_ood[0], preds_ood)
        
        # Compute global metrics under OOD evaluation paradigm
        aupr_OOD  = eval_ood.get_aupr_score(scores_test, scores_ood)
        auroc_OOD = eval_ood.get_auroc_score(scores_test, scores_ood)
        tnr95_OOD = eval_ood.get_tnr_frac_tpr(scores_test, scores_ood, 0.95)

        # Compute global metrics under OMS evaluation paradigm
        aupr_OMS = eval_oms.get_aupr_score(scores_test, scores_ood)
        auroc_OMS = eval_oms.get_auroc_score(scores_test, scores_ood)
        tnr95_OMS = eval_oms.get_tnr_frac_tpr(scores_test, scores_ood, 0.95)

        print("scores (test):\n%s" % (scores_test))
        print("scores (ood):\n%s" % str(scores_ood))

        data = [
            aupr_OOD, auroc_OOD, tnr95_OOD,
            aupr_OMS, auroc_OMS, tnr95_OMS,
        ]
        result[gmm_name] = data

... evaluating parameters (n=5, c=diag)
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
scores (test):
[-1622.04145234 -1619.27118423 -1583.18667683 ... -1609.56802142
 -1630.09724831 -1641.52132114]
scores (ood):
[-1674.77895963 -1673.87075439 -1669.20623543 ... -1420.67360692
 -1413.37914811 -1405.35773582]


In [10]:
df = pd.DataFrame.from_dict(result, orient='index')
df

,0,1,2,3,4,5
GMM_5_diag,0.845868,0.858373,0.4533,0.457084,0.74241,0.334898


## Experiment

Run the `Scripts/explo_gmm_script.py` script to get performance results of the GMM monitor with diverse hyper-parameters.

In [ ]:
%run Scripts/explo_gmm_script.py

In [ ]:
import csv
import os
import torch

from Monitors import GMMMonitor
from Monitors import MahalanobisMonitor
from evaluation import evaluate_monitor

import warnings
warnings.filterwarnings('ignore')


batch_size = 100
TORCH_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

all_networks = ["resnet", "densenet"]
all_networks_layers = [[32], [98]]
all_datasets = ["cifar10"]
all_datasets_ood = [["cifar100", "svhn", "lsun"]]
all_perturbations = ["brightness", "blur", "pixelization"]
all_adver_attacks = ["fgsm", "deepfool", "pgd"]

all_hp_n_components = ["auto_bic", "auto_knee"]
all_hp_t_covariance = ["full", "diag", "tied"]

path_to_save_results = "Results/GMM_study/"
if not os.path.exists(path_to_save_results):
    os.makedirs(path_to_save_results)

header = [
    "Network", "Network Layer",
    "Dataset", "Dataset OOD", "Perturbation", "Attack",
    "Monitor", "GMM n_components", "GMM t_covariance", "GMM architecture",
    "AUPR (OMS)", "AUROC (OMS)", "TNR @95 TPR (OMS)",
    "Train time", "Infer time"
]
config = {
    'network': None,
    'network_layers': None,
    'dataset': None,
    'dataset_ood': None,
    'monitor': None,
    'monitor_train_params': True,
    'monitor_infer_params': 'features',
    'perturbation': None,
    'adver_attack': None,
    'batch_size': batch_size,
    'TORCH_DEVICE': TORCH_DEVICE,
    'evaluation_mode': 'oms',
    'evaluation_metrics': ['aupr_score', 'auroc_score', 'tnr_frac_tpr'],
    'is_train_timed': True,
    'is_infer_timed': True,
}

for i_network in range(len(all_networks)):
    network = all_networks[i_network]
    network_layers = all_networks_layers[i_network]

    config['network'] = network
    config['network_layers'] = network_layers

    for i_dataset in range(len(all_datasets)):
        dataset = all_datasets[i_dataset]
        
        # Test with OOD as novelty
        for j_dataset in range(len(all_datasets_ood)):
            dataset_ood = all_datasets_ood[i_dataset][j_dataset]

            config['dataset'] = dataset
            config['dataset_ood'] = dataset_ood
            config['perturbation'] = None
            config['adver_attack'] = None

            print("Evaluating on %s, for dataset %s and OOD dataset %s." % (network, dataset, (dataset_ood, None, None)), flush=True)
            path_to_results_file = path_to_save_results + "%s_%s_%s.csv" % (network, dataset, dataset_ood)

            if (not os.path.exists(path_to_results_file)) or len(list(csv.reader(open(path_to_results_file)))) < 15:
                f = open(path_to_results_file, "w", encoding="UTF8")
                writer = csv.writer(f)
                writer.writerow(header)

                for hp_n in all_hp_n_components:
                    for hp_c in all_hp_t_covariance:
                        print("... HPs n=%s, c=%s" % (hp_n, hp_c), flush=True)
                        monitor = GMMMonitor(
                            dataset,
                            network,
                            network_layers[0],
                            n_components=hp_n,
                            t_covariance=hp_c)
                        
                        config['monitor'] = monitor
                        result = evaluate_monitor(config)

                        print("results:", result)
                        data = [
                            network, network_layers[0],
                            dataset, dataset_ood, str(None), str(None),
                            "GMM_%s_%s" % (hp_n, hp_c), hp_n, hp_c, 
                            str([gmm.n_components for gmm in monitor.gmm]),
                            result["aupr_score"],
                            result["auroc_score"],
                            result["tnr_frac_tpr"],
                            result["train_time"],
                            result["infer_time"]
                        ]
                        writer.writerow(data)
                f.close()

        # Test with OOD as cov shift
        for j in range(len(all_perturbations)):
            dataset_ood = dataset
            perturbation = all_perturbations[j]

            config['dataset'] = dataset
            config['dataset_ood'] = dataset_ood
            config['perturbation'] = perturbation
            config['adver_attack'] = None

            print("Evaluating on %s, for dataset %s and OOD dataset %s." % (network, dataset, (dataset_ood, perturbation, None)), flush=True)
            path_to_results_file = path_to_save_results + "%s_%s_%s.csv" % (network, dataset, perturbation)

            if (not os.path.exists(path_to_results_file)) or len(list(csv.reader(open(path_to_results_file)))) < 15:
                f = open(path_to_results_file, "w", encoding="UTF8")
                writer = csv.writer(f)
                writer.writerow(header)

                for hp_n in all_hp_n_components:
                    for hp_c in all_hp_t_covariance:
                        monitor = GMMMonitor(
                            dataset,
                            network,
                            network_layers[0],
                            n_components=hp_n,
                            t_covariance=hp_c)
                        
                        config['monitor'] = monitor
                        result = evaluate_monitor(config)

                        data = [
                            network, network_layers[0],
                            dataset, dataset_ood, perturbation, str(None),
                            "GMM_%s_%s" % (hp_n, hp_c), hp_n, hp_c, 
                            str([gmm.n_components for gmm in monitor.gmm]),
                            result["aupr_score"],
                            result["auroc_score"],
                            result["tnr_frac_tpr"],
                            result["train_time"],
                            result["infer_time"]
                        ]
                        writer.writerow(data)
                f.close()

        # Test with OOD as adversarial attack
        for j in range(len(all_adver_attacks)):
            dataset_ood = dataset
            adver_attack = all_adver_attacks[j]

            config['dataset'] = dataset
            config['dataset_ood'] = dataset_ood
            config['perturbation'] = None
            config['adver_attack'] = adver_attack

            print("Evaluating on %s, for dataset %s and OOD dataset %s." % (network, dataset, (dataset_ood, None, adver_attack)), flush=True)
            path_to_results_file = path_to_save_results + "%s_%s_%s.csv" % (network, dataset, adver_attack)

            if (not os.path.exists(path_to_results_file)) or len(list(csv.reader(open(path_to_results_file)))) < 15:
                f = open(path_to_results_file, "w", encoding="UTF8")
                writer = csv.writer(f)
                writer.writerow(header)

                for hp_n in all_hp_n_components:
                    for hp_c in all_hp_t_covariance:
                        monitor = GMMMonitor(
                            dataset,
                            network,
                            network_layers[0],
                            n_components=hp_n,
                            t_covariance=hp_c)
                        
                        config['monitor'] = monitor
                        result = evaluate_monitor(config)

                        data = [
                            network, network_layers[0],
                            dataset, dataset_ood, str(None), adver_attack,
                            "GMM_%s_%s" % (hp_n, hp_c), hp_n, hp_c, 
                            str([gmm.n_components for gmm in monitor.gmm]),
                            f'{result["aupr_score"]:.4f}',
                            f'{result["auroc_score"]:.4f}',
                            f'{result["tnr_frac_tpr"]:.4f}',
                            f'{result["train_time"]:.3f}',
                            f'{result["infer_time"]:.3f}'
                        ]
                        writer.writerow(data)
                f.close()